In [0]:
%run ./opal_functions

In [0]:
cursor = ""
page_size = 1000
ret = get_opal_groups(cursor=cursor, page_size=page_size)
groups_accum = ret.json()['results']
if ret.json()['next'] != None:
    cursor = ret.json()['next']
    while cursor != None:
        ret = get_opal_groups(cursor=cursor, page_size=page_size)
        groups_accum += ret.json()['results']
        if ret.json()['next'] != 'None':
            cursor = ret.json()['next']

In [0]:
updated = []
count = 1
for group in groups_accum:
    print(f'Processing group {count} of {len(groups_accum)}')
    new_row = group
    vis = get_opal_group_visibility(group['group_id'])
    if vis.status_code == 200:
        new_row['visibility'] = vis.json()['visibility']
        new_row['visibility_group_ids'] = vis.json()['visibility_group_ids']
        updated.append(new_row)
    else:
        print(f'Error getting visibility for group {group["group_id"]}')
    count += 1
groups_accum = updated

In [0]:
from pyspark.sql.functions import col, explode
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, ArrayType

schema = StructType([
    StructField("admin_owner_id", StringType(), True),
    StructField("app_id", StringType(), True),
    StructField("auto_approval", BooleanType(), True),
    StructField("description", StringType(), True),
    StructField("group_id", StringType(), True),
    StructField("group_type", StringType(), True),
    StructField("is_requestable", BooleanType(), True),
    StructField("metadata", StringType(), True),
    StructField("name", StringType(), True),
    StructField("remote_id", StringType(), True),
    StructField("recommended_duration", StringType(), True),
    StructField("request_configuration_list", 
        ArrayType(
            StructType([
                StructField("allow_requests", BooleanType(), True),            
                StructField("auto_approval", BooleanType(), True),
                StructField("priority", IntegerType(), True),
                StructField("recommended_duration_minutes", IntegerType(), True),
                StructField("require_mfa_to_request", BooleanType(), True),
                StructField("require_support_ticket", BooleanType(), True),
                StructField("reviewer_stages", 
                            ArrayType(
                                StructType([
                                    StructField("operator", StringType(), True),
                                    StructField("owner_ids",
                                                ArrayType(StringType(), True))
                                ]),
                                True
                            )),
                StructField("require_manager_approval", BooleanType(), True)  
            ])
        ), 
        True
    ),
    StructField("require_manager_approval", BooleanType(), True),
    StructField("visibility", StringType(), True),
    StructField("visibility_group_ids", ArrayType(StringType()), True)
])

df = spark.createDataFrame(groups_accum, schema)

df_formatted = df.select(
    "admin_owner_id",
    "app_id",
    col("auto_approval").alias("group_auto_approval"),
    "description",
    "group_id",
    "group_type",
    "is_requestable",
    "metadata",
    col("name").alias("group_name"),
    "remote_id",
    "recommended_duration",
    explode("request_configuration_list").alias("request")
).select(
    "admin_owner_id",
    "app_id",
    "group_auto_approval",
    "description",
    "group_id",
    "group_type",
    "is_requestable",
    "metadata",
    "group_name",
    "remote_id",
    "recommended_duration",
    "request.allow_requests",
    col("request.auto_approval").alias("request_auto_approval"),
    "request.priority",
    "request.recommended_duration_minutes",
    "request.require_mfa_to_request",
    "request.require_support_ticket",
    "request.require_manager_approval",
    explode("request.reviewer_stages").alias("reviewer_stage")
).select(
    "admin_owner_id",
    "app_id",
    "group_auto_approval",
    "description",
    "group_id",
    "group_type",
    "is_requestable",
    "metadata",
    "group_name",
    "remote_id",
    "recommended_duration",
    "allow_requests",
    "request_auto_approval",
    "priority",
    "recommended_duration_minutes",
    "require_mfa_to_request",
    "require_support_ticket",
    "require_manager_approval",
    "reviewer_stage.operator",
    explode("reviewer_stage.owner_ids").alias("reviewer_stages_owner_ids")
)

display(df_formatted)

In [0]:
df.write.option("mergeSchema", "true").saveAsTable("workspace.opal_dev.opal_demo_groups", mode="overwrite") 
df_formatted.write.option("mergeSchema", "true").saveAsTable("workspace.opal_dev.opal_demo_groups_formatted", mode="overwrite") 

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-3181193331971299>, line 1
----> 1 df.write.option("mergeSchema", "true").saveAsTable("workspace.opal_dev.opal_demo_groups", mode="overwrite") 
      2 df_formatted.write.option("mergeSchema", "true").saveAsTable("workspace.opal_dev.opal_demo_groups_formatted", mode="overwrite")

NameError: name 'df' is not defined